In [15]:
import requests
from bs4 import BeautifulSoup
import re

headers = {"User-Agent": "Mozilla/5.0"}

def clean_title(title):
    title = re.sub(r"\(.*?\)", "", title)
    title = re.sub(r"\[.*?\]", "", title)
    title = re.sub(r"\bfeat\.?\b.*", "", title, flags=re.I)
    title = re.sub(r"featuring.*", "", title, flags=re.I)
    return title.strip()

def scrape_year(year):
    url = f"https://kworb.net/spotify/songs_{year}.html"

    response = requests.get(url, headers=headers)
    response.encoding = "utf-8"
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    songs = []

    table = soup.find("table")
    if table is None:
        return songs

    for row in table.find_all("tr")[1:]:
        cols = row.find_all("td")

        if len(cols) < 2:
            continue

        artist_title = cols[0].get_text(strip=True)

        if " - " in artist_title:
            artist, title = artist_title.split(" - ", 1)

            songs.append({
                "year": year,
                "artist": artist.strip(),
                "title": clean_title(title)
            })

    return songs

In [16]:
all_songs = []

for year in range(2015, 2026):
    try:
        all_songs.extend(scrape_year(year))
    except requests.HTTPError:
        print(f"Could not retrieve {year}")

In [19]:
import pandas as pd

df = pd.DataFrame(all_songs)

pd.set_option('display.max_rows', None)
display(df)

df.to_csv("spotify_songs_2015_2025.csv", index=False, encoding="utf-8-sig") #if we need to export data as csv for other parts of the project

,year,artist,title
0,2015,Lord Huron,The Night We Met
1,2015,Justin Bieber,Love Yourself
2,2015,Twenty One Pilots,Stressed Out
3,2015,The Weeknd,The Hills
4,2015,Justin Bieber,Sorry
5,2015,Major Lazer,Lean On
6,2015,Charlie Puth,We Don't Talk Anymore
7,2015,Lukas Graham,7 Years
8,2015,Shawn Mendes,Stitches
9,2015,Tame Impala,The Less I Know The Better
